## Data Collection For System

In [ ]:
from google.colab import drive
import os
import pandas as pd

# Mount Drive
drive.mount('/content/drive')

# Đường dẫn tới folder mới của bạn
# Lưu ý: 'Books RecSys Dataset (API Call)' phải khớp chính xác tên folder trên Drive
FOLDER_PATH = '/content/drive/MyDrive/Books RecSys Dataset (API Call)'

if not os.path.exists(FOLDER_PATH):
    os.makedirs(FOLDER_PATH)
    print(f"✅ Đã kết nối và xác nhận folder: {FOLDER_PATH}")

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Books RecSys Dataset (API Call)/books.csv')
df.head()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6810 entries, 0 to 6809
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   isbn13          6810 non-null   int64  
 1   isbn10          6810 non-null   object 
 2   title           6810 non-null   object 
 3   subtitle        2381 non-null   object 
 4   authors         6738 non-null   object 
 5   categories      6711 non-null   object 
 6   thumbnail       6481 non-null   object 
 7   description     6548 non-null   object 
 8   published_year  6804 non-null   float64
 9   average_rating  6767 non-null   float64
 10  num_pages       6767 non-null   float64
 11  ratings_count   6767 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 638.6+ KB


In [ ]:
df.describe()

,isbn13,published_year,average_rating,num_pages,ratings_count
count,6.810000e+03,6804.000000,6767.000000,6767.000000,6.767000e+03
mean,9.780677e+12,1998.630364,3.933284,348.181026,2.106910e+04
std,6.068911e+08,10.484257,0.331352,242.376783,1.376207e+05
min,9.780002e+12,1853.000000,0.000000,0.000000,0.000000e+00
25%,9.780330e+12,1996.000000,3.770000,208.000000,1.590000e+02
50%,9.780553e+12,2002.000000,3.960000,304.000000,1.018000e+03
75%,9.780810e+12,2005.000000,4.130000,420.000000,5.992500e+03
max,9.789042e+12,2019.000000,5.000000,3342.000000,5.629932e+06


In [ ]:
df.isnull().sum()

,0
isbn13,0
isbn10,0
title,0
subtitle,4429
authors,72
categories,99
thumbnail,329
description,262
published_year,6
average_rating,43


In [ ]:
import requests
import time
import os
from tqdm import tqdm

# --- 1. CẤU HÌNH ---
INPUT_FILE = '/content/drive/MyDrive/Books RecSys Dataset (API Call)/books.csv' # Thay bằng tên file thực tế của bạn
OUTPUT_FILE = "final_clean_books.csv"
API_KEY = "YOUR_API_KEY"

def heal_book_data_safe(input_path, output_path, api_key):
    print(" Đang khởi động hệ thống Data Healer...")

    # Đọc dữ liệu (Hỗ trợ chạy tiếp nếu đã có file output)
    if os.path.exists(output_path):
        print(f" Nạp dữ liệu từ tiến độ cũ: {output_path}")
        df = pd.read_csv(output_path, low_memory=False)
    else:
        print(f" Nạp dữ liệu gốc từ: {input_path}")
        df = pd.read_csv(input_path, low_memory=False)

    # 2. XÁC ĐỊNH MỤC TIÊU (Smart Masking)
    # Kích hoạt gọi API nếu thiếu MỘT TRONG CÁC trường quan trọng này
    core_cols = ['description', 'categories', 'authors', 'thumbnail', 'average_rating', 'ratings_count']
    mask = df[core_cols].isnull().any(axis=1)
    indices_to_process = df[mask].index

    print(f" Tìm thấy {len(indices_to_process)} cuốn sách cần bổ sung dữ liệu cốt lõi.")

    base_url = "https://www.googleapis.com/books/v1/volumes"
    api_limit_reached = False

    # 3. VÒNG LẶP XỬ LÝ
    for count, idx in enumerate(tqdm(indices_to_process, desc="Calling API")):
        row = df.loc[idx]

        # Ưu tiên dùng isbn13, nếu không có thì dùng isbn10, cuối cùng là title
        query = ""
        if pd.notna(row.get('isbn13')):
            query = f"isbn:{str(row['isbn13']).replace('.0', '')}"
        elif pd.notna(row.get('isbn10')):
            query = f"isbn:{str(row['isbn10'])}"
        else:
            query = f"intitle:{str(row['title']).strip()}"

        try:
            params = {'q': query, 'key': api_key, 'maxResults': 1}
            resp = requests.get(base_url, params=params, timeout=10)

            # --- CẦU DAO AN TOÀN ---
            if resp.status_code in [429, 403]:
                print(f"\n🛑 CẢNH BÁO ĐỎ: Đã vượt quá Quota của Google (Mã {resp.status_code})!")
                print("Hệ thống tự động ngắt kết nối để bảo vệ dữ liệu.")
                api_limit_reached = True
                break

            if resp.status_code == 200:
                data = resp.json()

                if 'items' in data:
                    info = data['items'][0].get('volumeInfo', {})

                    # Lấp đầy các giá trị NẾU ô hiện tại đang trống (NaN)
                    if pd.isna(row.get('description')):
                        df.at[idx, 'description'] = info.get('description', 'Not Found')

                    if pd.isna(row.get('categories')):
                        df.at[idx, 'categories'] = ", ".join(info.get('categories', []))

                    if pd.isna(row.get('authors')):
                        df.at[idx, 'authors'] = ", ".join(info.get('authors', []))

                    if pd.isna(row.get('thumbnail')):
                        df.at[idx, 'thumbnail'] = info.get('imageLinks', {}).get('thumbnail', '')

                    if pd.isna(row.get('subtitle')):
                        df.at[idx, 'subtitle'] = info.get('subtitle', '')

                    # Lấy dữ liệu số (Ratings & Pages)
                    if pd.isna(row.get('average_rating')) and 'averageRating' in info:
                        df.at[idx, 'average_rating'] = info['averageRating']

                    if pd.isna(row.get('ratings_count')) and 'ratingsCount' in info:
                        df.at[idx, 'ratings_count'] = info['ratingsCount']

                    if pd.isna(row.get('num_pages')) and 'pageCount' in info:
                        df.at[idx, 'num_pages'] = info['pageCount']

                    # Trích xuất năm xuất bản (publishedDate thường có dạng YYYY hoặc YYYY-MM-DD)
                    if pd.isna(row.get('published_year')) and 'publishedDate' in info:
                        year = str(info['publishedDate'])[:4] # Cắt lấy 4 số đầu
                        if year.isdigit():
                            df.at[idx, 'published_year'] = int(year)
                else:
                    # Nếu Google không có thông tin, đánh dấu để không gọi lại lần sau
                    if pd.isna(row.get('description')): df.at[idx, 'description'] = 'Not Found'

            time.sleep(0.3) # Giữ nhịp độ an toàn

        except Exception as e:
            # Nếu lỗi mạng (Timeout), nghỉ 2 giây rồi chạy tiếp
            time.sleep(2)
            continue

        # Tự động lưu sau mỗi 50 cuốn (vì dataset nhỏ nên ta lưu thường xuyên hơn)
        if (count + 1) % 50 == 0:
            df.to_csv(output_path, index=False)

    # 4. CHỐT FILE VÀ BÁO CÁO
    df.to_csv(output_path, index=False)

    if api_limit_reached:
        print("\n⚠️ Đã lưu tiến độ trước khi ngắt. Bạn có thể đổi Key và chạy lại.")
    else:
        print(f"\n✅ Xử lý thành công toàn bộ dữ liệu! File chốt: {output_path}")

# --- THỰC THI ---
heal_book_data_safe(INPUT_FILE, OUTPUT_FILE, API_KEY)

 Đang khởi động hệ thống Data Healer...
 Nạp dữ liệu từ tiến độ cũ: final_clean_books.csv
 Tìm thấy 530 cuốn sách cần bổ sung dữ liệu cốt lõi.


Calling API: 100%|██████████| 530/530 [03:13<00:00,  2.74it/s]



✅ Xử lý thành công toàn bộ dữ liệu! File chốt: final_clean_books.csv


In [ ]:
df_final = pd.read_csv("/content/final_clean_books.csv")

In [ ]:
df_final.head()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [ ]:
df_final.isnull().sum()

,0
isbn13,0
isbn10,0
title,0
subtitle,4417
authors,58
categories,79
thumbnail,286
description,169
published_year,4
average_rating,37


In [ ]:
df.isnull().sum()

,0
isbn13,0
isbn10,0
title,0
subtitle,4429
authors,72
categories,99
thumbnail,329
description,262
published_year,6
average_rating,43


In [ ]:
total_missing_rows = df_final.isnull().any(axis=1).sum()
print(f"Total rows with at least one missing value: {total_missing_rows}")

total_missing_rows = df.isnull().any(axis=1).sum()
print(f"Total rows with at least one missing value: {total_missing_rows}")

Total rows with at least one missing value: 4582
Total rows with at least one missing value: 4628


In [ ]:
# --- CẤU HÌNH ---
# Nạp lại file mà Google Books API vừa bó tay (file bạn vừa lưu xong)
INPUT_FILE = "final_clean_books.csv"
OUTPUT_FILE = "books_fully_healed.csv"

def open_library_sweeper(input_path, output_path):
    print("🧹 Kích hoạt Open Library API - Đội cứu hộ dữ liệu...")
    df = pd.read_csv(input_path, low_memory=False)

    # 1. TÌM KIẾM CÁC DÒNG CẦN CỨU
    # Lấy những dòng mà Google trả về 'Not Found' hoặc đang bị trống Description
    mask = (df['description'] == 'Not Found') | (df['description'].isnull())
    indices_to_heal = df[mask].index

    print(f" Tìm thấy {len(indices_to_heal)} cuốn sách bị Google từ chối. Chuyển cho Open Library...")

    base_url = "https://openlibrary.org/api/books"

    # 2. VÒNG LẶP GỌI OPEN LIBRARY
    for count, idx in enumerate(tqdm(indices_to_heal, desc="Open Library Scanning")):
        row = df.loc[idx]

        # Open Library bắt buộc phải dùng ISBN (không tìm bằng title được như Google)
        isbn = None
        if pd.notna(row.get('isbn13')):
            isbn = str(row['isbn13']).replace('.0', '').strip()
        elif pd.notna(row.get('isbn10')):
            isbn = str(row['isbn10']).strip()

        if not isbn:
            continue # Nếu không có cả ISBN10 và 13 thì Open Library cũng bó tay

        # Tạo query key
        bib_key = f"ISBN:{isbn}"
        params = {
            'bibkeys': bib_key,
            'format': 'json',
            'jscmd': 'data' # Lấy toàn bộ data chi tiết
        }

        try:
            resp = requests.get(base_url, params=params, timeout=10)

            if resp.status_code == 200:
                data = resp.json()

                # Nếu API trả về dữ liệu cho cuốn ISBN này
                if bib_key in data:
                    book_info = data[bib_key]

                    # 1. Cứu Description (Open Library lưu khá lộn xộn, có lúc ở 'description', có lúc ở 'notes')
                    desc = book_info.get('description', '')
                    if isinstance(desc, dict): # Đôi khi nó là 1 dict {'type': '...', 'value': '...'}
                        desc = desc.get('value', '')
                    if not desc: # Fallback sang trường notes nếu description trống
                        desc = book_info.get('notes', '')

                    if desc:
                        df.at[idx, 'description'] = desc

                    # 2. Cứu Author
                    if pd.isna(row.get('authors')) or row.get('authors') == '':
                        authors_list = book_info.get('authors', [])
                        if authors_list:
                            df.at[idx, 'authors'] = ", ".join([a.get('name', '') for a in authors_list])

                    # 3. Cứu Thumbnail (Cover)
                    if pd.isna(row.get('thumbnail')) or row.get('thumbnail') == '':
                        cover = book_info.get('cover', {})
                        if 'medium' in cover:
                            df.at[idx, 'thumbnail'] = cover['medium']
                        elif 'large' in cover:
                            df.at[idx, 'thumbnail'] = cover['large']

                    # 4. Cứu số trang (Number of pages)
                    if pd.isna(row.get('num_pages')) or row.get('num_pages') == 0:
                        if 'number_of_pages' in book_info:
                            df.at[idx, 'num_pages'] = book_info['number_of_pages']

            # Open Library là server phi lợi nhuận (Internet Archive),
            # hãy gọi chậm lại 1 chút (0.5s) để tôn trọng hệ thống của họ và không bị block IP
            time.sleep(0.5)

        except Exception as e:
            time.sleep(2)
            continue

        # Lưu định kỳ mỗi 50 cuốn
        if (count + 1) % 50 == 0:
            df.to_csv(output_path, index=False)

    # 3. CHỐT FILE TỔNG HỢP CUỐI CÙNG
    df.to_csv(output_path, index=False)
    print(f"\n✅ Đợt rà soát thứ 2 hoàn tất! Dữ liệu đã lưu tại: {output_path}")

# --- THỰC THI ---
open_library_sweeper(INPUT_FILE, OUTPUT_FILE)

🧹 Kích hoạt Open Library API - Đội cứu hộ dữ liệu...
 Tìm thấy 221 cuốn sách bị Google từ chối. Chuyển cho Open Library...


Open Library Scanning: 100%|██████████| 221/221 [04:15<00:00,  1.15s/it]



✅ Đợt rà soát thứ 2 hoàn tất! Dữ liệu đã lưu tại: books_fully_healed.csv


In [ ]:
df_final_2 = pd.read_csv("/content/books_fully_healed.csv")
df_final_2.head()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [ ]:
df_final.isnull().sum()

,0
isbn13,0
isbn10,0
title,0
subtitle,4417
authors,58
categories,79
thumbnail,286
description,169
published_year,4
average_rating,37


In [ ]:
df_final_2.isnull().sum()

,0
isbn13,0
isbn10,0
title,0
subtitle,4417
authors,53
categories,79
thumbnail,215
description,117
published_year,4
average_rating,37


In [ ]:
import shutil
import os

source_file = '/content/books_fully_healed.csv'
destination_folder = FOLDER_PATH # This variable is already defined in the notebook

# Ensure the destination folder exists
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

destination_file = os.path.join(destination_folder, 'books_fully_healed.csv')

try:
    shutil.move(source_file, destination_file)
    print(f"✅ Successfully moved '{source_file}' to '{destination_file}'")
except FileNotFoundError:
    print(f"❌ Error: Source file '{source_file}' not found.")
except Exception as e:
    print(f"❌ An error occurred while moving the file: {e}")

✅ Successfully moved '/content/books_fully_healed.csv' to '/content/drive/MyDrive/Books RecSys Dataset (API Call)/books_fully_healed.csv'
